In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

### 파일명: 생성형 AI.ipynb, 생성형AI.html, 실행화면.png을 압축하여 

### "본인이름_생성형AI.zip으로 메일첨부해 주세요 

# 5. 생성형 AI 평가 :

- 첫번째 체인 : 나라이름 -> 그나라에서 가장 유명한 음식  # food_chain
- 두번째 체인 : 음식 -> 음식의 레시피 # recipe_chain
- 최종 체인: 나라이름 -> 그 나라의 가장 유명한 음식의 레시피# final_chain 

In [2]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
import os
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field
llm=ChatOllama(model='llama3.2:1b')

In [3]:
# 첫번째 체인 
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt_template = PromptTemplate(
    template='What is the most famous food in that {country}? Return the name of the food only.',
    input_variables=['country']
)
outputParser =StrOutputParser()
outputParser.invoke(llm.invoke(prompt_template.invoke({'country':'Korea'})))
food_chain = prompt_template|llm | outputParser
food_chain.invoke({'country':'Korea'})

'Bibimbap'

In [4]:
# 두 번째 체인 (recipe_chain): 음식 -> 음식의 레시피
recipe_prompt = PromptTemplate(
    template="""Write a clear recipe for this food:{food}
Please show me the easiest recipe I've ever seen.""",
    input_variables=["food"]
)
recipe_chain = recipe_prompt | llm | outputParser
result=recipe_chain.invoke({'food':'Bulgogi'})
print(result)

Bulgogi is a popular Korean dish that consists of marinated beef, typically thinly sliced, grilled or stir-fried. Here's a simple recipe for you:

**Easiest Bulgogi Recipe Ever**

**Ingredients:**

* 1 pound (450g) thinly sliced beef ( ribeye or sirloin), cut into thin strips
* 1/2 cup (120ml) soy sauce
* 1/4 cup (60ml) honey
* 2 tablespoons (30g) Gochujang (Korean chili paste)
* 2 cloves garlic, minced
* 1 tablespoon (15g) grated fresh ginger
* 2 tablespoons (30g) vegetable oil
* 2 green onions, chopped (optional)
* Sesame seeds and chopped cilantro for garnish (optional)

**Instructions:**

1. **Prepare the marinade:** In a large bowl, whisk together soy sauce, honey, Gochujang, garlic, and ginger.
2. **Add the beef:** Add the sliced beef to the marinade and toss to coat evenly. Cover the bowl with plastic wrap and refrigerate for at least 2 hours or overnight.
3. **Preheat the grill or pan:** Heat a grill or large skillet over medium-high heat (or a non-stick pan over medium heat) f

In [5]:
# 최종 체인 (final_chain): 나라이름 -> 그 나라의 가장 유명한 음식의 레시피
from langchain_core.runnables import RunnableLambda
final_chain = food_chain | (lambda food_name: {"food": food_name}) | recipe_chain
result = final_chain.invoke({"country": "Korea"})
print(result)

Bibimbap, a classic Korean dish! I'd be happy to provide you with a simple recipe. Here's the easiest way to make Bibimbap:

**Ingredients:**

* 1 cup of cooked white or brown rice (preferably day-old rice)
* 1/2 cup of mixed vegetables (e.g., bean sprouts, zucchini, carrots, green onions)
* 1/2 cup of diced Korean chili flakes (gochugaru)
* 2 eggs, fried or poached
* 1/4 cup of chopped fresh cilantro or scallions
* 2 tablespoons of Gochujang (Korean chili paste)
* 1 tablespoon of soy sauce
* 1 tablespoon of rice vinegar
* Salt and pepper to taste
* Sesame oil and sesame seeds for garnish (optional)

**Instructions:**

1. **Prepare the vegetables:** Wash and chop the mixed vegetables into bite-sized pieces.
2. **Prepare the egg:** If using fried eggs, heat a non-stick pan over medium heat and crack in the egg. Cook until the whites are set and the yolks are still runny. If poaching eggs, bring a pot of water to a boil and gently place the eggs in. Cook for 3-4 minutes, then transfer to